# Bit rate, bandwidth and capacity {#sec-bitrate}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/optoelectronics-ch2/blob/main/notebooks/bitrate.ipynb){.colab-link} &nbsp;Run this chapter's code in Google Colab (no installation).

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/optoelectronics-ch2/blob/main/notebooks/bitrate.ipynb)

> Generated from the Quarto companion. The browser sliders of the website are not included; see `interactive_explorers.ipynb` for ipywidgets versions.

In [ ]:
# --- setup: works locally and in Google Colab ---------------------------
import sys
if 'google.colab' in sys.modules:
    import urllib.request
    urllib.request.urlretrieve('https://raw.githubusercontent.com/philippevelha/optoelectronics-ch2/main/fiberlib.py', 'fiberlib.py')
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'OptiCommPy'])
else:
    sys.path.insert(0, '..')                 # fiberlib.py is in the parent folder
def ojs_define(**kw): pass                   # only used by the Quarto website
# ----------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import fiberlib as fl
fl.use_style()
c0, PS = fl.c0, fl.PS_NM_KM

*Slides 96–121.* Dispersion stretches each bit. When neighbouring bits overlap too much, the receiver can no longer
tell a "1" from a "0": this is **intersymbol interference (ISI)**. Noise and attenuation then decide how much
overlap is tolerable. This chapter connects the fiber physics to the bit error rate.

## Signals, spectra and limits (slides 97–105)

**Nyquist:** a noiseless channel of bandwidth $B$ carries at most $2B$ binary symbols per second
(more with multi-level symbols). **Shannon:** with noise,

$$
C=B\log_2\left(1+\frac{S}{N}\right).
$$

```{ojs}
//| echo: false
viewof sh = Inputs.form({
  B: Inputs.number({value: 3200, label: "bandwidth B (Hz)"}),
  snr: Inputs.range([-10, 60], {value: 30, step: 1, label: "S/N (dB)"})
})
```

```{ojs}
//| echo: false
{
  const s = 10 ** (sh.snr / 10), C = sh.B * Math.log2(1 + s);
  return html`<div class="result-box">S/N = ${s.toFixed(1)} → C = <b>${C >= 1e9 ? (C/1e9).toFixed(2)+" Gb/s" : C >= 1e6 ? (C/1e6).toFixed(2)+" Mb/s" : (C/1e3).toFixed(2)+" kb/s"}</b>
  · Nyquist binary limit 2B = ${(2*sh.B/1e3).toFixed(1)} kb/s · levels needed ≈ √(1+S/N) = ${Math.sqrt(1+s).toFixed(1)}</div>`
}
```

With the defaults you get the slide 104 example (31.9 kb/s on a 3.2 kHz line at 30 dB). For a single
C-band channel ($B\approx50$ GHz, $S/N\approx20$ dB), Shannon allows about 330 Gb/s per polarisation.
Commercial coherent transceivers now operate within a few dB of this limit.

In [ ]:
#| label: fig-snr
#| fig-cap: "The same NRZ bit pattern received with S/N = 1, 10 and 100 (cf. slide 105)."
rng = np.random.default_rng(0)
bits = fl.prbs(7, 24)
x = np.repeat(bits, 40).astype(float)
fig, axs = plt.subplots(3, 1, figsize=(8, 4.6), sharex=True)
for ax, snr in zip(axs, [1, 10, 100]):
    noise = rng.normal(0, np.sqrt(np.mean(x**2) / snr), x.size)
    ax.plot(x + noise, color=fl.PALETTE[0], lw=0.8); ax.plot(x, color=fl.INK, lw=1.2)
    ax.set_ylabel(f"S/N={snr}"); ax.set_yticks([])
axs[-1].set_xlabel("sample"); plt.tight_layout(); plt.show()

## Line codes: RZ and NRZ (slides 96, 120)

In [ ]:
#| label: fig-codes
#| fig-cap: "The same bits in RZ and NRZ format. NRZ needs half the bandwidth for the same bit rate, while RZ is more robust to ISI and easier for clock recovery."
bits = np.array([0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0])
sps = 50
t = np.arange(len(bits) * sps) / sps
nrz = np.repeat(bits, sps)
rz = nrz * (np.tile(np.arange(sps), len(bits)) < sps // 2)
fig, axs = plt.subplots(2, 1, figsize=(8, 3.2), sharex=True)
axs[0].plot(t, rz, color=fl.PALETTE[1]); axs[0].set_ylabel("RZ")
axs[1].plot(t, nrz, color=fl.PALETTE[0]); axs[1].set_ylabel("NRZ")
for ax in axs:
    ax.set_yticks([0, 1]); ax.set_xticks(range(len(bits) + 1))
    for i, b in enumerate(bits): ax.text(i + .5, 1.15, str(b), ha="center", fontsize=8, color=fl.INK2)
    ax.set_ylim(-0.1, 1.35)
axs[1].set_xlabel("time (bit periods)"); plt.tight_layout(); plt.show()

## Dispersion-limited bit rate (slides 115–121)

For a Gaussian output pulse with rms width $\sigma$ ($\Delta\tau_{1/2}=2.355\sigma$, i.e. $\sigma\approx0.425\Delta\tau_{1/2}$),
allowing the overlap shown on slide 116:

$$
B_\text{RZ}\approx\frac{0.25}{\sigma}=\frac{0.59}{\Delta\tau_{1/2}},\qquad B_\text{NRZ}\approx 2B_\text{RZ}\ (\text{slide 96}),
\qquad f_\text{op}\approx0.75B,\quad f_\text{el}\approx0.70f_\text{op}.
$$

In [ ]:
D, dl, L = 8, 2, 10                  # ps/nm/km, nm, km
t12_per_km = D * dl                  # ps/km
BL = 0.59 / (t12_per_km * 1e-12)     # b/s·km
B = BL / L
print(f"Slide 121: Δτ½/L = {t12_per_km} ps/km → BL = {BL/1e9:.1f} Gb/s·km; "
      f"for 10 km B = {B/1e9:.2f} Gb/s, f_op = {0.75*B/1e9:.2f} GHz, f_el = {0.7*0.75*B/1e9:.2f} GHz")

### Two regimes: source linewidth vs. signal bandwidth (new)

The rule $\Delta\tau=|D|L\Delta\lambda$ assumes the **source** is broad ($\Delta\lambda$ fixed by the laser).
Modern DFB lasers are so narrow that the spectrum is set by the **modulation itself**, $\Delta\lambda\approx\lambda^2B/c$.
Then $\Delta\tau\approx|D|L\lambda^2B/c$, and requiring $\Delta\tau\lesssim1/B$ (one bit slot) gives the famous

$$
B^2L\lesssim\frac{c}{|D|\lambda^2}\quad\Rightarrow\quad\text{10 Gb/s: } \sim\!70\text{ km},\quad\text{40 Gb/s: } \sim\!5\text{ km at 17 ps/nm/km}.
$$

Doubling the bit rate divides the reach by **four**.

In [ ]:
#| label: fig-reach
#| fig-cap: "Dispersion-limited reach versus bit rate for standard SMF (17 ps/nm/km) at 1550 nm. Broad sources (dashed) give B·L = const. Narrow lasers give B²·L = const."
Bs = np.logspace(9, 11.3, 100)
fig, ax = plt.subplots()
for k, dl in enumerate([1.0, 0.1]):
    ax.loglog(Bs / 1e9, 0.59 / (17e-12 * dl) / Bs * 2, "--", color=fl.PALETTE[k], label=f"source Δλ = {dl} nm (NRZ)")
ax.loglog(Bs / 1e9, c0 / (17 * PS * (1.55e-6) ** 2 * Bs**2) / 1e3, color=fl.INK, label="transform-limited, B²L rule")
ax.set_xlabel("bit rate (Gb/s)"); ax.set_ylabel("max. length (km)"); ax.legend(fontsize=8)
plt.show()

## Simulating a real link (slides 72–73)

Now we put everything together in a minimal **intensity-modulation / direct-detection (IM/DD)** simulator,
`fl.imdd_link` (about 40 lines of numpy):

1. PRBS bits → NRZ drive with finite rise time and extinction ratio,
2. chirp-free optical field $E=\sqrt{P(t)}$,
3. fiber: $\tilde E(\omega,L)=\tilde E(\omega,0)\,e^{j\beta_2\omega^2L/2}\,10^{-\alpha L/20}$, with $\beta_2=-D\lambda^2/2\pi c$,
4. PIN photodiode $I=\mathcal R|E|^2$ plus shot and thermal noise, then an electrical filter of bandwidth $0.7B$,
5. sampling, $Q=(\mu_1-\mu_0)/(\sigma_1+\sigma_0)$ and $\text{BER}=\tfrac12\operatorname{erfc}(Q/\sqrt2)$.

Parameters of slide 73: 10 Gb/s, 3 dBm launch, NRZ, D = 17 ps/nm/km, 0.2 dB/km, no amplifier.

In [ ]:
#| label: fig-eyes
#| fig-cap: "Eye diagrams after 0, 50, 100 and 130 km (10 Gb/s NRZ, 3 dBm, no amplification). Both the received power (loss) and ISI (dispersion) close the eye."
bits = fl.prbs(9, 2000)
sps = 32
fig, axs = plt.subplots(1, 4, figsize=(12, 3), sharey=False)
for ax, L in zip(axs, [0, 50, 100, 130]):
    t, Ptx, I = fl.imdd_link(bits, L_km=L, sps=sps)
    Q, ber, ph = fl.eye_metrics(I, bits, sps)
    seg = 2 * sps
    start = ph                     # eye opening centred at t = 1 bit
    for k in range(20, 400):
        s = I[start + k * sps: start + k * sps + seg]
        if len(s) == seg: ax.plot(np.arange(seg) / sps, s * 1e3, color=fl.PALETTE[0], lw=0.4, alpha=0.25)
    ax.set_title(f"{L} km  Q = {Q:.1f}  BER ≈ {ber:.0e}", fontsize=9)
    ax.set_xlabel("time (bits)")
axs[0].set_ylabel("photocurrent (mA)")
plt.tight_layout(); plt.show()

### Separating loss and dispersion

In [ ]:
#| label: fig-ber-L
#| fig-cap: "BER versus distance at 10 Gb/s. Without an amplifier (blue) the link fails at ≈ 100 km. With the received power held constant by an ideal amplifier, dispersion alone (orange) still limits it to ≈ 80 km at BER 10⁻⁹ (≈ 120 km with FEC), consistent with the B²L rule. Halving the bit rate (green) extends the dispersion limit ~4×."
Ls = np.arange(0, 251, 10)
def ber_vs_L(**kw):
    out = []
    for L in Ls:
        _, _, I = fl.imdd_link(bits, L_km=L, sps=16, **kw)
        out.append(max(fl.eye_metrics(I, bits, 16)[1], 1e-16))
    return np.array(out)
fig, ax = plt.subplots()
ax.semilogy(Ls, ber_vs_L(), "o-", ms=4, label="10 Gb/s, loss + dispersion")
ax.semilogy(Ls, ber_vs_L(alpha_dB_km=0.0, P_launch_dBm=-12), "s-", ms=4, label="10 Gb/s, dispersion only (P_rx = −12 dBm)")
ax.semilogy(Ls, ber_vs_L(alpha_dB_km=0.0, P_launch_dBm=-12, Rb=5e9), "^-", ms=4, label="5 Gb/s, dispersion only")
ax.axhline(1e-9, color=fl.INK2, lw=0.8, ls=":"); ax.text(250, 2e-9, "BER 10⁻⁹", ha="right", color=fl.INK2)
ax.axhline(3.8e-3, color=fl.INK2, lw=0.8, ls="--"); ax.text(250, 5e-3, "HD-FEC limit", ha="right", color=fl.INK2)
ax.set_ylim(1e-16, 1); ax.set_xlabel("L (km)"); ax.set_ylabel("BER"); ax.legend(fontsize=8)
plt.show()

### Why IM/DD and dispersion don't mix: power fading (new)

Square-law detection of a chirp-free, dispersed signal acts as a filter on the intensity:
$|H(f)|=|\cos(2\pi^2\beta_2Lf^2)|$. Its first null is at $f_1=1/\sqrt{4\pi|\beta_2|L}$. Once $f_1$ falls inside the
signal band, whole frequency components disappear and no amount of power will bring them back.

In [ ]:
#| label: fig-fading
#| fig-cap: "Small-signal intensity transfer function of SMF (17 ps/nm/km) for several lengths. The 10 Gb/s NRZ main lobe (shaded) is hit by the first null around 100 km."
f = np.linspace(0, 20e9, 800)
b2 = -17 * PS * (1.55e-6) ** 2 / (2 * np.pi * c0)
fig, ax = plt.subplots()
ax.axvspan(0, 7, color=fl.PALETTE[0], alpha=0.07, lw=0)
for k, L in enumerate([20, 50, 100, 200]):
    ax.plot(f / 1e9, 20 * np.log10(np.abs(np.cos(2 * np.pi**2 * b2 * L * 1e3 * f**2)) + 1e-6), color=fl.PALETTE[k], label=f"{L} km")
ax.set_ylim(-30, 2); ax.set_xlabel("modulation frequency (GHz)"); ax.set_ylabel("|H(f)|² (dB)"); ax.legend()
plt.show()
for L in [50, 100, 200]:
    print(f"L = {L} km: first null at {1/np.sqrt(4*np.pi*abs(b2)*L*1e3)/1e9:.1f} GHz")

## Live link simulator (runs in your browser)

The same IM/DD model, written in JavaScript, runs in real time. Change the distance, bit rate or launch power
and watch the eye.

```{ojs}
//| echo: false
viewof link = Inputs.form({
  L:   Inputs.range([0, 200], {value: 60, step: 1, label: "length (km)"}),
  Rb:  Inputs.select([2.5, 5, 10, 25, 40], {value: 10, label: "bit rate (Gb/s)"}),
  D:   Inputs.range([-20, 20], {value: 17, step: 0.5, label: "D (ps nm⁻¹ km⁻¹)"}),
  a:   Inputs.range([0, 0.5], {value: 0.2, step: 0.01, label: "loss (dB/km)"}),
  P:   Inputs.range([-10, 15], {value: 3, step: 0.5, label: "launch power (dBm)"}),
  fmt: Inputs.radio(["NRZ", "RZ"], {value: "NRZ", label: "format"}),
  amp: Inputs.toggle({label: "ideal preamplifier (fix P_rx = −10 dBm)", value: false})
})
```

```{ojs}
//| echo: false
function fft(re, im, inv) {          // in-place radix-2
  const n = re.length;
  for (let i = 1, j = 0; i < n; i++) { let bit = n >> 1; for (; j & bit; bit >>= 1) j ^= bit; j ^= bit;
    if (i < j) { [re[i], re[j]] = [re[j], re[i]]; [im[i], im[j]] = [im[j], im[i]]; } }
  for (let len = 2; len <= n; len <<= 1) {
    const ang = 2 * Math.PI / len * (inv ? 1 : -1), wr = Math.cos(ang), wi = Math.sin(ang);
    for (let i = 0; i < n; i += len) { let cr = 1, ci = 0;
      for (let k = 0; k < len / 2; k++) {
        const ar = re[i+k], ai = im[i+k], br = re[i+k+len/2]*cr - im[i+k+len/2]*ci, bi = re[i+k+len/2]*ci + im[i+k+len/2]*cr;
        re[i+k] = ar + br; im[i+k] = ai + bi; re[i+k+len/2] = ar - br; im[i+k+len/2] = ai - bi;
        const t = cr * wr - ci * wi; ci = cr * wi + ci * wr; cr = t; } } }
  if (inv) for (let i = 0; i < n; i++) { re[i] /= n; im[i] /= n; }
}
sim = {
  const nb = 256, sps = 16, N = nb * sps, Rb = link.Rb * 1e9, fs = sps * Rb, c = 299792458, lam = 1550e-9;
  let st = 0x7f; const bits = [];
  for (let i = 0; i < nb; i++) { const b = ((st >> 6) ^ (st >> 5)) & 1; bits.push(st & 1); st = ((st << 1) | b) & 0x7f; }
  const freq = i => (i < N/2 ? i : i - N) * fs / N;
  // drive with Gaussian rise time 0.3 T
  let re = new Float64Array(N), im = new Float64Array(N);
  for (let i = 0; i < N; i++) { const k = Math.floor(i / sps), ph = i % sps;
    re[i] = bits[k] * (link.fmt === "RZ" ? (ph < sps/2 ? 1 : 0) : 1); }
  fft(re, im, false);
  const sg = 0.3 / Rb / 2.563;
  for (let i = 0; i < N; i++) { const g = Math.exp(-0.5 * (2*Math.PI*freq(i)*sg)**2); re[i] *= g; im[i] *= g; }
  fft(re, im, true);
  const er = 10 ** (-1.5); let mean = 0;
  for (let i = 0; i < N; i++) { re[i] = er + (1 - er) * Math.min(Math.max(re[i], 0), 1); mean += re[i] / N; }
  const P0 = 1e-3 * 10 ** (link.P / 10);
  for (let i = 0; i < N; i++) { re[i] = Math.sqrt(re[i] / mean * P0); im[i] = 0; }
  // fiber
  fft(re, im, false);
  const b2 = -link.D * 1e-6 * lam * lam / (2 * Math.PI * c), Lm = link.L * 1e3;
  const att = link.amp ? 1 : 10 ** (-link.a * link.L / 20);
  for (let i = 0; i < N; i++) { const w = 2 * Math.PI * freq(i), phi = b2 / 2 * w * w * Lm;
    const r = re[i], m = im[i]; re[i] = (r * Math.cos(phi) - m * Math.sin(phi)) * att; im[i] = (r * Math.sin(phi) + m * Math.cos(phi)) * att; }
  fft(re, im, true);
  let I = new Float64Array(N), Pm = 0;
  for (let i = 0; i < N; i++) { I[i] = re[i]*re[i] + im[i]*im[i]; Pm += I[i] / N; }
  if (link.amp) { const g = 1e-4 / Pm; for (let i = 0; i < N; i++) I[i] *= g; Pm = 1e-4; }
  // receiver: R = 0.9 A/W, shot + thermal (50 ohm, 3 dB NF), Bel = 0.7 Rb
  const q = 1.602e-19, kT = 1.38e-23 * 300, Bel = 0.7 * Rb;
  let nre = new Float64Array(N), nim = new Float64Array(N);
  for (let i = 0; i < N; i++) {
    const Ii = 0.9 * I[i], s2 = 2*q*Ii*Bel + 4*kT*Bel/50*2;
    const u1 = Math.random() || 1e-12, u2 = Math.random();
    nre[i] = Ii + Math.sqrt(-2*Math.log(u1)) * Math.cos(2*Math.PI*u2) * Math.sqrt(s2 * fs / (2*Bel));
  }
  fft(nre, nim, false);
  for (let i = 0; i < N; i++) { const g = Math.exp(-Math.LN2 / 2 * (freq(i) / Bel) ** 2); nre[i] *= g; nim[i] *= g; }
  fft(nre, nim, true);
  // best sampling phase
  let best = {Q: -1, ph: 0};
  for (let ph = 0; ph < sps; ph++) {
    const one = [], zero = [];
    for (let k = 4; k < nb - 4; k++) (bits[k] ? one : zero).push(nre[k * sps + ph]);
    const Q = (d3.mean(one) - d3.mean(zero)) / (d3.deviation(one) + d3.deviation(zero));
    if (Q > best.Q) best = {Q, ph};
  }
  const traces = [];
  for (let k = 4; k < nb - 6; k++) for (let j = 0; j <= 2 * sps; j++) {
    const idx = k * sps + best.ph - sps + j;
    traces.push({t: j / sps, I: nre[idx] * 1e3, k});
  }
  return {traces, Q: best.Q, Prx: 10 * Math.log10(Pm / 1e-3)};
}
```

```{ojs}
//| echo: false
Plot.plot({
  width: Math.min(width, 1050), height: 330, x: {label: "time (bit periods)", domain: [0, 2]}, y: {label: "photocurrent (mA)"},
  marks: [Plot.line(sim.traces, {x: "t", y: "I", z: "k", stroke: "#2a78d6", strokeOpacity: 0.18, strokeWidth: 1}),
          Plot.ruleX([1], {stroke: "#e34948", strokeDasharray: "4,3"})]
})
```

```{ojs}
//| echo: false
{
  const ber = 0.5 * erfc(sim.Q / Math.SQRT2);
  const Lrule = 299792458 / (Math.abs(link.D) * 1e-6 * (1550e-9)**2 * (link.Rb*1e9)**2) / 1e3;   // B²L rule, km
  return html`<div class="result-box">P<sub>rx</sub> = ${sim.Prx.toFixed(1)} dBm · Q = <b>${sim.Q.toFixed(2)}</b> ·
   BER ≈ <b>${ber < 1e-15 ? "< 1e-15" : ber.toExponential(1)}</b>
   ${ber < 1e-9 ? "✅ error-free" : ber < 3.8e-3 ? "⚠️ needs FEC" : "❌ link down"}
   · B²L-rule reach c/(|D|λ²B²) ≈ ${link.D === 0 ? "∞" : Lrule.toFixed(0) + " km"}</div>`
}
function erfc(x) {          // Numerical Recipes erfc, |rel err| < 1.2e-7
  const z = Math.abs(x), t = 1 / (1 + 0.5 * z);
  const r = t * Math.exp(-z*z - 1.26551223 + t*(1.00002368 + t*(0.37409196 + t*(0.09678418 + t*(-0.18628806 +
    t*(0.27886807 + t*(-1.13520398 + t*(1.48851587 + t*(-0.82215223 + t*0.17087277)))))))));
  return x >= 0 ? r : 2 - r;
}
```

::: {.callout-tip}
## Things to try
- Set D to **−17**. Does the eye care about the sign? (Not for a chirp-free source. With a chirped, directly
  modulated laser it would.)
- Find the maximum distance at 40 Gb/s. Then switch on the preamplifier. Which effect limits the link now?
- Compare RZ and NRZ at 100 km.
:::

## Going further with OptiCommPy

The notebook `opticomm.ipynb` uses [OptiCommPy](https://github.com/edsonportosilva/OptiCommPy), a full
research-grade library. Since version 0.10 the pulse-shaping API takes a `parameters` object, which is why some
cells of the original notebook fail. The corrected minimal version below gives the same Q as our simulator.

In [ ]:
#| code-fold: show
from optic.models.devices import mzm, photodiode
from optic.models.channels import linearFiberChannel
from optic.comm.modulation import modulateGray
from optic.comm.metrics import bert
from optic.dsp.core import firFilter, pulseShape, upsample, pnorm
from optic.utils import parameters, dBm2W

SpS, Rs = 16, 10e9
pp = parameters(); pp.pulseType, pp.SpS = "nrz", SpS          # new API (>= 0.10)
pulse = pulseShape(pp); pulse /= abs(pulse).max()
pM = parameters(); pM.Vpi, pM.Vb = 2, -1
pC = parameters(); pC.alpha, pC.D, pC.Fc, pC.Fs = 0.2, 16, 193.1e12, SpS * Rs
pP = parameters(); pP.ideal, pP.B, pP.Fs, pP.seed = False, Rs, SpS * Rs, 1

np.random.seed(123); bitsTx = np.random.randint(2, size=20000)
sigTx = firFilter(pulse, upsample(pnorm(modulateGray(bitsTx, 2, "pam")), SpS))
for L in [0, 50, 100]:
    pC.L = L
    I = photodiode(linearFiberChannel(mzm(np.sqrt(dBm2W(3)), sigTx, pM), pC), pP)
    BER, Q = bert(I[0::SpS] / np.std(I), bitsTx)
    print(f"L = {L:3d} km: Q = {Q:5.2f}, BER = {BER:.1e}")

## Try it yourself

1. Add a **chirp** parameter to `fl.imdd_link` ($E=\sqrt P\,e^{j\alpha_H\ln P/2}$, Henry factor $\alpha_H\approx3$–5) and show that the
   sign of $D$ now matters.
2. Replace the PIN with an **APD** (gain $M$, excess noise $F=M^x$). Find the optimum gain at 100 km.
3. Replace NRZ by **PAM-4** (2 bits/symbol, 4 levels, used in 400G data-centre links). What do you gain in
   dispersion tolerance, and what do you lose in sensitivity?